# Ejercicio 1 — Visualización del fraud_dataset

Inspección rápida del dataset de fraude para entender:
- Distribución del target `big_model_fraud_probability` (lo que debe replicar el TinyModel).
- Relación de cada feature con el target — scatter coloreado por `flagged_fraud` (ground truth, no usar en training).
- Detección visual de outliers, escalas dispares y posibles features que necesiten transformación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "../data and documentation/fraud_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(f"Filas: {len(df)}, Columnas: {df.shape[1]}")
df.head()

In [ ]:
df.describe()

## Distribución del target

`big_model_fraud_probability` debe ser float en `[0, 1]`. Histograma + comparación con `flagged_fraud`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["big_model_fraud_probability"], bins=50, edgecolor="black", alpha=0.75)
axes[0].set_xlabel("big_model_fraud_probability")
axes[0].set_ylabel("frecuencia")
axes[0].set_title("Distribución del target (BigModel)")

for cls in (0, 1):
    subset = df.loc[df["flagged_fraud"] == cls, "big_model_fraud_probability"]
    axes[1].hist(subset, bins=50, alpha=0.6, label=f"flagged_fraud={cls}")
axes[1].set_xlabel("big_model_fraud_probability")
axes[1].set_ylabel("frecuencia")
axes[1].set_title("Target separado por ground truth")
axes[1].legend()

plt.tight_layout()
plt.show()

## Scatter: cada feature vs target

Color = `flagged_fraud` (azul = legítimo, rojo = fraude real). Subsample para que no se sature el plot.

In [ ]:
FEATURES = [
    "timestamp",
    "amount_usd",
    "quantity_purchased",
    "session_duration_seconds",
    "days_since_last_purchase",
    "account_age_days",
    "device_screen_resolution",
    "time_since_last_login_s",
    "items_viewed_before_purchase",
]
TARGET = "big_model_fraud_probability"

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(df), size=min(3000, len(df)), replace=False)
sample = df.iloc[sample_idx]
colors = sample["flagged_fraud"].map({0: "#1f77b4", 1: "#d62728"})

n = len(FEATURES)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()

for ax, feat in zip(axes, FEATURES):
    ax.scatter(sample[feat], sample[TARGET], c=colors, s=8, alpha=0.4)
    ax.set_xlabel(feat)
    ax.set_ylabel(TARGET)
    ax.set_title(f"{feat} vs target")

for ax in axes[len(FEATURES):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Scatter en escala log para features con cola larga

Algunas features (montos, tiempos, resolución) suelen tener distribución muy sesgada; verlas en log ayuda a descubrir estructura oculta.

In [ ]:
LOG_FEATURES = [
    "amount_usd",
    "session_duration_seconds",
    "account_age_days",
    "device_screen_resolution",
    "time_since_last_login_s",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, feat in zip(axes, LOG_FEATURES):
    x = sample[feat].clip(lower=1e-3)
    ax.scatter(x, sample[TARGET], c=colors, s=8, alpha=0.4)
    ax.set_xscale("log")
    ax.set_xlabel(f"{feat} (log)")
    ax.set_ylabel(TARGET)
    ax.set_title(f"{feat} vs target (log x)")

for ax in axes[len(LOG_FEATURES):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Correlación lineal feature ↔ target

Pearson da una primera idea de qué features podría capturar un perceptrón lineal.

In [ ]:
corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
corr.plot.barh(ax=ax, color=["#d62728" if v < 0 else "#1f77b4" for v in corr])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson r vs big_model_fraud_probability")
ax.set_title("Correlación lineal de cada feature con el target")
plt.tight_layout()
plt.show()

corr